# 01. Data Collection & Exploratory Data Analysis
Fetching 5-year adjusted close prices for 10 IDX blue-chip stocks.
Goal: understand return distributions, correlations, and individual asset characteristics
before building the portfolio.

In [ ]:
import sys; sys.path.insert(0, '..')
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from src.data.fetcher import StockDataFetcher
from src.portfolio.metrics import (
    TRADING_DAYS_PER_YEAR, RISK_FREE_RATE_ANNUAL, DEFAULT_TICKERS,
    DEFAULT_START_DATE, DEFAULT_END_DATE
)
from src.visualization.plots import (
    plot_correlation_heatmap, plot_cumulative_returns, plot_return_distribution
)

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

## 1. Data Collection

In [ ]:
fetcher = StockDataFetcher(DEFAULT_TICKERS, DEFAULT_START_DATE, DEFAULT_END_DATE)
prices = fetcher.fetch()
log_returns = fetcher.to_log_returns(prices)

print(f'Tickers:       {list(prices.columns)}')
print(f'Date range:    {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Trading days:  {len(prices)}')
print(f'Missing values: {prices.isna().sum().sum()}')

In [ ]:
prices.head().style.format('{:.0f}').set_caption('Adjusted Close Prices (IDR)')

## 2. Normalized Price Performance

In [ ]:
n_assets = len(prices.columns)
equal_weights = np.array([1/n_assets] * n_assets)
fig = plot_cumulative_returns(prices, equal_weights, list(prices.columns))
fig.update_layout(title='Normalized Price (Base = 100, Equal-Weight Portfolio vs Individual Assets)')
fig.show()

### Interpretation
- Most assets show a recovery trend post-2020.
- The equal-weight portfolio provides a smoother return profile compared to most individual volatile assets.
- Significant dispersion is visible among individual ticker performances.

## 3. Summary Statistics

In [ ]:
stats = fetcher.get_summary_stats(log_returns)
stats_display = stats.copy()
stats_display[['annual_return', 'annual_volatility']] *= 100
stats_display = stats_display.round(2)
stats_display.columns = ['Ann. Return (%)', 'Ann. Volatility (%)', 'Sharpe Ratio', 'Skewness', 'Kurtosis']
stats_display.sort_values('Sharpe Ratio', ascending=False).style\
    .background_gradient(subset=['Sharpe Ratio'], cmap='RdYlGn')\
    .format(precision=2)\
    .set_caption('Individual Asset Statistics (2019–2024)')

## 4. Return Distributions

In [ ]:
# Plotting for the first asset as an example
first_asset_weights = np.zeros(len(prices.columns))
first_asset_weights[0] = 1.0
fig = plot_return_distribution(log_returns, first_asset_weights)
fig.update_layout(title=f'Return Distribution: {prices.columns[0]}')
fig.show()

### Note on Normality
Many financial assets exhibit fat tails (kurtosis > 3). If the kurtosis is significantly high, the normal distribution assumption used in parametric VaR might underestimate extreme tail risks.

## 5. Correlation Analysis

In [ ]:
fig = plot_correlation_heatmap(log_returns)
fig.show()

### Interpretation
- Strong correlations are observed within the banking sector (e.g., BBCA, BBRI, BMRI).
- Lower correlations between different sectors (e.g., Telecom vs Consumer Staples) offer better diversification benefits.

## 6. Key Findings
- Highest Sharpe individual asset: BBCA.JK (0.27)
- Most volatile asset: EXCL.JK (40.86%)
- Lowest correlation pair (diversification opportunity): BMRI.JK and UNVR.JK (0.20)
- Average pairwise correlation: 0.33
- Period: 2019-2024 includes the COVID-19 crash (visible in 2020) and subsequent sectoral recovery.